In [3]:
#/root/jupyter_notebooks/mvm-accelerator/test.ipynb

import pynq
from pynq import Overlay, allocate, MMIO, PL
from threading import Thread, Barrier
import numpy as np
import time, re

### Config

In [4]:
VERBOSE = 2 # (0, 1, 2, 3)
PROFILE = 1 # (0, 1)

In [5]:
# the following params are fixed in hardware

CDMA_BASE = 0xA0000000 
BRAM_BASE = 0x80000000 

AXIS_CLK_HZ = 166e6

AVAILABLE_CHANNELS = 4
ACTIVE_CHANNELS = AVAILABLE_CHANNELS

NUM_ROWS = 16384
ROWS_PER_CHANNEL = NUM_ROWS // ACTIVE_CHANNELS

WORD_WIDTH = 128
ELEMENT_WIDTH = 16
ELEMENTS_PER_WORD = WORD_WIDTH // ELEMENT_WIDTH
BYTES_PER_WORD = WORD_WIDTH // 8

ELEMENTS_PER_ROW = 16384
WORDS_PER_ROW = ELEMENTS_PER_ROW // ELEMENTS_PER_WORD
BYTES_PER_ROW = WORDS_PER_ROW * BYTES_PER_WORD

NUM_PARTITIONS = AVAILABLE_CHANNELS

ELEMENTS_PER_PARTITION = ELEMENTS_PER_ROW // NUM_PARTITIONS
WORDS_PER_PARTITION = WORDS_PER_ROW // NUM_PARTITIONS
BYTES_PER_PARTITION = WORDS_PER_PARTITION * BYTES_PER_WORD

if PROFILE: NUM_DMAS = AVAILABLE_CHANNELS

In [6]:
assert WORD_WIDTH % ELEMENT_WIDTH == 0
assert ELEMENTS_PER_ROW % ACTIVE_CHANNELS == 0
assert NUM_ROWS % ACTIVE_CHANNELS == 0

In [7]:
#PROJ_DIR  = f"/home/ubuntu/mvm-accelerator/hw/fp{ELEMENT_WIDTH}_{NUM_ROWS}x{ELEMENTS_PER_ROW}/"
PROJ_DIR  = f"/home/ubuntu/mvm-accelerator/hw/"
BIT       = PROJ_DIR + "design_1.bit"
MTRX_PATH = f"/home/ubuntu/mvm-accelerator/trmult_reduced{ELEMENT_WIDTH}.bin"

### Helpers

In [8]:
#https://github.com/iamNCJ/PYNQ-CDMA-Driver/blob/main/pynq_cdma/cdma.py

from pynq import DefaultIP
from pynq.buffer import PynqBuffer

class CDMA(DefaultIP):
    def __init__(self, description):
        super().__init__(description=description)

    bindto = ['xilinx.com:ip:axi_cdma:4.1']

    @property
    def idle(self):
        """
        `transfer` can only be called when the DMA is idle
        :return: True if the DMA engine is idle
        """
        return self.register_map.CDMASR.Idle

    def _do_transfer(self, source: int, dest: int, bytes_count: int):
        """
        Execute DMA transfer
        :param source: source address
        :param dest: destination address
        :param bytes_count: bytes to transfer
        :return: None
        """
        if not self.idle:
            raise Exception('DMA transfer can only start when engine is idle')

        assert source > 0
        if source > 0xFFFF_FFFF:
            self.register_map.SA_MSB[:] = source >> 8
        self.register_map.SA[:] = source & 0xFFFF_FFFF

        assert dest > 0
        if dest > 0xFFFF_FFFF:
            self.register_map.DA_MSB[:] = dest >> 8
        self.register_map.DA[:] = dest & 0xFFFF_FFFF

        assert bytes_count > 0
        self.register_map.BTT = bytes_count

        # Spin
        while not self.idle:
            pass

    def transfer(self, source, dest, bytes_count=None):
        """
        Transfer data through DMA
        :param source:
        :param dest:
        :param bytes_count:
        :return:
        """
        # Set source
        bytes_source = None
        if isinstance(source, PynqBuffer):
            source_addr = source.device_address
            bytes_source = source.nbytes
        elif isinstance(source, int):
            source_addr = source
        else:
            raise TypeError(f'Type {type(source)} not supported as source')

        bytes_dest = None
        if isinstance(dest, PynqBuffer):
            dest_addr = dest.device_address
            bytes_dest = dest.nbytes
        elif isinstance(dest, int):
            dest_addr = dest
        else:
            raise TypeError(f'Type {type(source)} not supported as dest')

        if bytes_count is None:
            if bytes_source is not None and bytes_dest is not None:
                bytes_count = min(bytes_source, bytes_dest)
            elif bytes_source is not None:
                bytes_count = bytes_source
            elif bytes_dest is not None:
                bytes_count = bytes_dest
            else:
                raise RuntimeError(f'Bytes to transfer is not set and cannot be inferred')

        self._do_transfer(source_addr, dest_addr, bytes_count)

In [9]:
from typing import Tuple, Dict, Any

class AxisDmaProfiler:
    """
    Address map:
    
        +0x00: STATUS [0]=have_result, [1]=running
        
        +0x04: CYCLES    [31:0]   +0x08: CYCLES    [63:32]
        +0x0C: BEATS     [31:0]   +0x10: BEATS     [63:32]
        +0x14: BYTES     [31:0]   +0x18: BYTES     [63:32]
        +0x1C: PKTS      [31:0]
        
        +0x20: READY_LOW [31:0]   +0x24: READY_LOW [63:32]
        +0x28: VALID_LOW [31:0]   +0x2C: VALID_LOW [63:32]
        
        +0x30: GAP_LAST  [31:0]   +0x34: GAP_LAST  [63:32]
        +0x38: GAP_TOTAL [31:0]   +0x3C: GAP_TOTAL [63:32]
        +0x40: GAP_MIN   [31:0]   +0x44: GAP_MIN   [63:32]
        +0x48: GAP_MAX   [31:0]   +0x4C: GAP_MAX   [63:32]
        +0x50: GAP_COUNT [31:0]
        
        +0x54: GAP_STATUS [0]=gap_valid, [1]=armed
        
        +0x58: BUSY_TOTAL[31:0]   +0x5C: BUSY_TOTAL[63:32]
    """
    
    OFF_STATUS = 0x00
    OFF_CYCLES = 0x04
    OFF_BEATS  = 0x0C
    OFF_BYTES  = 0x14
    OFF_PKTS   = 0x1C
    
    OFF_RDY_LOW = 0x20
    OFF_VLD_LOW = 0x28
    
    OFF_GAP_LAST   = 0x30
    OFF_GAP_TOTAL  = 0x38
    OFF_GAP_MIN    = 0x40
    OFF_GAP_MAX    = 0x48
    OFF_GAP_COUNT  = 0x50
    OFF_GAP_STATUS = 0x54
    OFF_BUSY_TOTAL = 0x58
    
    CH_WINDOW = 0x100 # 0x100 bytes per port

    def __init__(self, overlay, ports=4, axis_clk_hz=300e6, ip_name_contains="axis_dma_profiler"):
        
        self.overlay = overlay
        self.ports = ports
        self.axis_clk_hz = axis_clk_hz

        key = next((k for k in overlay.ip_dict if ip_name_contains in k), None)
        
        if key is None:
            raise ValueError(
                f"Could not find IP containing '{ip_name_contains}' in overlay.ip_dict. "
                f"Pass base=... explicitly or adjust ip_name_contains."
            )
        ip = overlay.ip_dict[key]
        base = int(ip["phys_addr"])
        size = int(ip["addr_range"])

        if base is None: raise ValueError("Profiler base address is unknown.")
        if size is None: raise ValueError("Profiler size is unknown.")

        self.base = base
        self.size = size
        self.mmio = MMIO(self.base, self.size)
        
    def _rd32(self, off):
        return self.mmio.read(off)

    def _rd64_stable(self, off, attempts=4):
        for _ in range(attempts):
            hi1 = self._rd32(off + 4)
            lo  = self._rd32(off)
            hi2 = self._rd32(off + 4)
            if hi1 == hi2:
                return (hi1 << 32) | lo
        return -1

    def _port_base(self, port):
        if not (0 <= port < self.ports):
            raise ValueError(f"Port out of range: {port} (ports={self.ports})")
        return port * self.CH_WINDOW

    # ========== Public ==========

    def stats(self, port) -> Dict[str, int]:
        """
        Reads (cycles, beats, bytes, pkts). Prefer calling when have_result=1 and running=0.
        """
        b = self._port_base(port)
        cycles = self._rd64_stable(b + self.OFF_CYCLES)
        beats  = self._rd64_stable(b + self.OFF_BEATS)
        bytes_ = self._rd64_stable(b + self.OFF_BYTES)
        pkts   = self._rd32(b + self.OFF_PKTS)
        
        rdy_lo = self._rd64_stable(b + self.OFF_RDY_LOW)
        vld_lo = self._rd64_stable(b + self.OFF_VLD_LOW)
        
        return {
            "cycles": cycles, 
            "beats":  beats, 
            "bytes":  bytes_, 
            "pkts":   pkts,
            "rdy_lo": rdy_lo,
            "vld_lo": vld_lo
        }
    
    def gap_stats(self, port) -> Dict[str, Any]:
        """
        Inter-packet gap metrics.
        """
        b = self._port_base(port)
        gap_last   = self._rd64_stable(b + self.OFF_GAP_LAST)
        gap_total  = self._rd64_stable(b + self.OFF_GAP_TOTAL)
        gap_min    = self._rd64_stable(b + self.OFF_GAP_MIN)
        gap_max    = self._rd64_stable(b + self.OFF_GAP_MAX)
        gap_count  = self._rd32(b + self.OFF_GAP_COUNT)
        busy_total = self._rd64_stable(b + self.OFF_BUSY_TOTAL)
        
        if gap_count == 0: gap_last = gap_total = gap_min = gap_max = None

        return {
            "gap_last":   gap_last,
            "gap_total":  gap_total,
            "gap_min":    gap_min,
            "gap_max":    gap_max,
            "gap_count":  gap_count,
            "busy_total": busy_total
        }

    def throughput(self, port) -> Tuple[float, float, Dict[str, int]]:
        """
        Returns (gbps, handshake_utilization, stats) for a completed frame .
        - gbps  = (bytes * 8 / cycles) * axis_clk_hz / 1e9
        - handshake_utilization = beats / cycles   (fraction of cycles that handshook)
        """
        s = self.stats(port)
        cycles = max(1, s["cycles"])        
        gbps = (s["bytes"] * 8.0 * self.axis_clk_hz / cycles) / 1e9
        util = s["beats"] / cycles
        return gbps, util, s
    
    def gap_summary(self, port):
        """
        Analogous to `throughput`: return (g, mean_gap_ns, util_long_run).
          - g: dict with gap aggregates (gap_last, gap_min/max, gap_total, gap_count, busy_total)
          - mean_gap_ns: average inter-packet gap in nanoseconds
          - util_long_run: busy_total / (busy_total + gap_total)
        """
        g = self.gap_stats(port)
        gap_total  = g["gap_total"]
        gap_count  = g["gap_count"]
        busy_total = g["busy_total"]

        mean_gap_cycles = (gap_total / gap_count) if gap_count else 0.0
        mean_gap_ns     = (1e9 * mean_gap_cycles / self.axis_clk_hz) if self.axis_clk_hz else 0.0
        util_long_run   = (busy_total / (busy_total + gap_total)) if gap_total else 0.0

        return g, int(mean_gap_cycles), int(mean_gap_ns), util_long_run

    def __repr__(self) -> str:
        return f"AxisDmaProfiler(base=0x{self.base:08x}, size=0x{self.size:x}, ports={self.ports})"


In [10]:
def create_workers(barrier, dmas, matrix, result):
    workers = []
    for ch in range(ACTIVE_CHANNELS-1):
        w = Thread(target=worker, args=(barrier, ch, dmas[ch], matrix[ch], result[ch]))
        workers.append(w)
        
    return workers

In [11]:
def worker(barrier, ch, dma, matrix, result):    
        
    # arm result channel
    dma.recvchannel.transfer(result)
    if VERBOSE > 1: print(f"[WORKER {ch}] Starting up...")

    # start timer
    barrier.wait()
    if ch == ACTIVE_CHANNELS-1: 
        start = time.perf_counter()
    barrier.wait()
    
    # send inputs
    dma.sendchannel.transfer(matrix)
    dma.sendchannel.wait()
    if VERBOSE > 1: print(f"[WORKER {ch}] Inputs sent.")

    # wait for output
    dma.recvchannel.wait()
    if VERBOSE > 1: print(f"[WORKER {ch}] Results received.")
    result.invalidate()
        
    # end timer
    barrier.wait()
    if ch == ACTIVE_CHANNELS - 1: 
        end = time.perf_counter()
        return (end - start) * 1000

# Main

### Initialize (run once)

In [12]:
if VERBOSE: 
    print(f"\n================================ Parameters =================================")

    print(f"""
CDMA_BASE = 0x{CDMA_BASE:08x}
BRAM_BASE = 0x{BRAM_BASE:08x}

NUM_ROWS           = {NUM_ROWS}
AVAILABLE_CHANNELS = {AVAILABLE_CHANNELS} 
ACTIVE_CHANNELS    = {ACTIVE_CHANNELS}
ROWS_PER_CHANNEL   = {ROWS_PER_CHANNEL}

WORD_WIDTH        = {WORD_WIDTH}
ELEMENT_WIDTH     = {ELEMENT_WIDTH}
ELEMENTS_PER_WORD = {ELEMENTS_PER_WORD}
BYTES_PER_WORD    = {BYTES_PER_WORD}

ELEMENTS_PER_ROW = {ELEMENTS_PER_ROW}
WORDS_PER_ROW    = {WORDS_PER_ROW}
BYTES_PER_ROW    = {BYTES_PER_ROW}

NUM_PARTITIONS = {NUM_PARTITIONS}

ELEMENTS_PER_PARTITION = {ELEMENTS_PER_PARTITION}
WORDS_PER_PARTITION    = {WORDS_PER_PARTITION}
BYTES_PER_PARTITION    = {BYTES_PER_PARTITION}""")

matrix = vector = result = None 

if VERBOSE: print("\n================================== Setup ====================================\n")

# Load overlay
if VERBOSE: print("Loading Overlay.")
overlay = Overlay(BIT, download=True)
if not overlay.is_loaded():
    raise RuntimeError("Overlay download failed.")
        
# Allocate buffers
if VERBOSE: print("Allocating buffers.")
matrix = [allocate(shape=(ROWS_PER_CHANNEL, ELEMENTS_PER_ROW), dtype=np.float16) for _ in range(ACTIVE_CHANNELS)]
vector = allocate(shape=(ELEMENTS_PER_ROW,), dtype=np.float16)
result = allocate(shape=(ACTIVE_CHANNELS, ROWS_PER_CHANNEL), dtype=np.float64)

if VERBOSE > 1:
    for ch, m in enumerate(matrix):
        print(f"    matrix[{ch}]: addr=0x{m.physical_address:08x}, nbytes=0x{m.nbytes:08x}")
    print(f"    vector: addr=0x{vector.physical_address:08x}, nbytes=0x{vector.nbytes:08x}")
    print(f"    result: addr=0x{result.physical_address:08x}, nbytes=0x{result.nbytes:08x}")

if VERBOSE: print("\n================================ Initialize =================================\n")
    
# Initialize buffers
if VERBOSE: print("Initializing Buffers.")
for m in matrix: 
    m[:] = np.random.rand(ROWS_PER_CHANNEL, ELEMENTS_PER_ROW).astype(np.float16)
    m.flush()  
vector[:] = np.random.rand(ELEMENTS_PER_ROW).astype(np.float16)
vector.flush()
result.fill(0.0)
    
# Initialize profiler
if PROFILE: 
    if VERBOSE: print("Initializing Profiler.")
    prof = AxisDmaProfiler(overlay, ports=NUM_DMAS, axis_clk_hz=AXIS_CLK_HZ)
    if VERBOSE > 1: print(f"    {prof.__repr__()}")

# Initialize DMAs
if VERBOSE: print("Initializing DMAs.")
dmas = [
    getattr(overlay, name)
    for name in sorted(overlay.ip_dict, key=lambda n: int(re.search(r"\d+$", n).group()))
    if name.startswith("axi_dma")
]
if VERBOSE > 1: print("    dmas =", [d.description['fullpath'] for d in dmas])
if len(dmas) != AVAILABLE_CHANNELS:
    raise RuntimeError(f"Check overlay. Found {len(dmas)} DMAs with AVAILABLE_CHANNELS={AVAILABLE_CHANNELS}.")
        
if VERBOSE: print("\n=============================================================================\n")


================================ Parameters =================================

CDMA_BASE = 0xa0000000
BRAM_BASE = 0x80000000

NUM_ROWS           = 16384
AVAILABLE_CHANNELS = 4 
ACTIVE_CHANNELS    = 4
ROWS_PER_CHANNEL   = 4096

WORD_WIDTH        = 128
ELEMENT_WIDTH     = 16
ELEMENTS_PER_WORD = 8
BYTES_PER_WORD    = 16

ELEMENTS_PER_ROW = 16384
WORDS_PER_ROW    = 2048
BYTES_PER_ROW    = 32768

NUM_PARTITIONS = 4

ELEMENTS_PER_PARTITION = 4096
WORDS_PER_PARTITION    = 512
BYTES_PER_PARTITION    = 8192

================================== Setup ====================================

Loading Overlay.


Allocating buffers.
    matrix[0]: addr=0x39300000, nbytes=0x08000000
    matrix[1]: addr=0x41300000, nbytes=0x08000000
    matrix[2]: addr=0x49300000, nbytes=0x08000000
    matrix[3]: addr=0x51300000, nbytes=0x08000000
    vector: addr=0x375b0000, nbytes=0x00008000
    result: addr=0x375c0000, nbytes=0x00020000

================================ Initialize =================================

Initializing Buffers.
Initializing Profiler.
    AxisDmaProfiler(base=0xa00c0000, size=0x1000, ports=4)
Initializing DMAs.
    dmas = ['axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'axi_dma_3']




### Execute

In [23]:
NUM_RUNS = 4

for n in range(NUM_RUNS):
    if VERBOSE: print(f"\n============================== Send Vector ({n}) ==============================\n")

    overlay.download()
        
    # Write vector to BRAM
    if VERBOSE: print("Writing vector to BRAM.")
    cdma = CDMA(overlay.ip_dict['axi_cdma_0'])
    if VERBOSE > 1: 
        cdma_addr = hex(overlay.ip_dict['axi_cdma_0']['phys_addr'])
        print(f"    cdma@{cdma_addr}.transfer(src=0x{vector.physical_address:08x}, dest=0x{BRAM_BASE:08x})")
        
    vec_start = time.perf_counter()
    cdma.transfer(vector, BRAM_BASE)
    vec_end = time.perf_counter()

    vec_time = (vec_end - vec_start) * 1000
    print(f"Elapsed time: {vec_time:.6f} ms")

    if VERBOSE: print(f"\n================================ Compute ({n}) ================================\n")

    # Create worker threads
    barrier = Barrier(ACTIVE_CHANNELS)
    workers = create_workers(barrier, dmas, matrix, result)
    last_ch = ACTIVE_CHANNELS-1

    # Aaaand... Go!    
    for w in workers: w.start()
    comp_time = worker(barrier, last_ch, dmas[last_ch], matrix[last_ch], result[last_ch]) # we only have 4 hardware threads
    for w in workers: w.join()

    # Print results
    if VERBOSE:
        if VERBOSE > 1: print()
        for i in range(ACTIVE_CHANNELS):
            for j in range(ROWS_PER_CHANNEL):
                row_idx = i * ROWS_PER_CHANNEL + j
                if (row_idx > 5 and row_idx < NUM_ROWS-5):
                    if not VERBOSE > 2:
                        if (row_idx == 6): print("...")
                        continue
                actual = result[i][j]
                expected = float(np.dot(matrix[i][j], vector.astype(np.float64)))
                print(f"Row {row_idx}: Actual={actual:.32f} | Expected={expected:.32f}")

    print(f"\n============================== Performance ({n}) ==============================\n")

    if PROFILE:
        for i in range(prof.ports):
            gbps, util, s = prof.throughput(port=i)
            g, mean_gap_cycles, mean_gap_ns, util_long_run = prof.gap_summary(port=i)
            print(f"Port {i}:")
            print(f"    Handshake util: {util:.3%} | Throughput: {gbps/8:.3f} GB/s")
            if g["gap_count"] > 0: print(f"    Avg gap len: {mean_gap_cycles} cycles, {mean_gap_ns} ns | Busy util: {util_long_run:.2%}")
            if VERBOSE > 1:
                print(f"    {s}")
                if g["gap_count"] > 0: print(f"    {g}")
        print()

    print(f"Compute time: {comp_time:.6f} ms")

print("\n=============================================================================\n")


============================== Send Vector (0) ==============================

Writing vector to BRAM.
    cdma@0xa0000000.transfer(src=0x375b0000, dest=0x80000000)
Elapsed time: 38.893809 ms

================================ Compute (0) ================================

[WORKER 0] Starting up...
[WORKER 1] Starting up...
[WORKER 2] Starting up...
[WORKER 3] Starting up...
[WORKER 1] Inputs sent.
[WORKER 0] Inputs sent.
[WORKER 0] Results received.
[WORKER 2] Inputs sent.
[WORKER 1] Results received.
[WORKER 2] Results received.
[WORKER 3] Inputs sent.
[WORKER 3] Results received.

Row 0: Actual=4131.35928940773010253906250000000000 | Expected=4131.37018750752031337469816207885742
Row 1: Actual=4147.48509997129440307617187500000000 | Expected=4147.49692489912558812648057937622070
Row 2: Actual=4132.04232865571975708007812500000000 | Expected=4132.05181478205486200749874114990234
Row 3: Actual=4095.65649342536926269531250000000000 | Expected=4095.64866374061602982692420482635498
Row 4:

### Cleanup

In [24]:
if matrix is not None: [m.freebuffer() for m in matrix]
for buf in [vector, result]: 
    if buf is not None: buf.freebuffer()